# Exploratory Data Analysis: M5 Retail Demand & Intermittency
This notebook explores the core statistical characteristics of the M5 Walmart dataset that drove every modeling decision:
1. **Trend & Level Analysis**
2. **Multi-Scale Seasonality** (Weekly, Annual, SNAP Benefits, Holiday closures)
3. **Demand Categorization** (Syntetos-Boylan ADI vs $CV^2$ Intermittency Framework)
4. **Price Elasticity & Promotions**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Load processed fact table or raw sample
try:
    df = pd.read_parquet('../data/processed/fact_sales.parquet')
except Exception:
    print("Run `python src/pipeline.py` or generate data first.")

## 1. Demand Intermittency Classification (ADI vs $CV^2$)
In retail supply chain, demand patterns are categorized into 4 quadrants based on:
- **ADI (Average Demand Interval)**: Average periods between non-zero sales (Threshold: 1.32)
- **$CV^2$**: Squared coefficient of variation of non-zero quantities (Threshold: 0.49)

- **Smooth**: Regular demand, low variance (suits standard time series)
- **Intermittent**: Sporadic demand, constant size (suits Croston or Tweedie GBDT)
- **Erratic**: Regular demand, highly volatile sizes
- **Lumpy**: Sporadic demand, highly volatile sizes (hardest to forecast)

In [ ]:
def classify_demand(series):
    non_zeros = series[series > 0]
    if len(non_zeros) <= 1:
        return pd.Series({'ADI': 99.0, 'CV2': 99.0, 'category': 'Lumpy'})
    
    # Interval between non-zeros
    adi = len(series) / len(non_zeros)
    
    # CV^2 of non-zero demand
    cv2 = (non_zeros.std() / (non_zeros.mean() + 1e-6)) ** 2
    
    if adi < 1.32 and cv2 < 0.49:
        cat = 'Smooth'
    elif adi >= 1.32 and cv2 < 0.49:
        cat = 'Intermittent'
    elif adi < 1.32 and cv2 >= 0.49:
        cat = 'Erratic'
    else:
        cat = 'Lumpy'
        
    return pd.Series({'ADI': adi, 'CV2': cv2, 'category': cat})

if 'df' in locals():
    intermittency_stats = df.groupby('id')['units_sold'].apply(classify_demand).unstack()
    print(intermittency_stats['category'].value_counts(normalize=True) * 100)

## 2. Weekly & Annual Seasonality
Exploring weekend uplift and holiday effects (e.g. Christmas shutdowns and SNAP days).

In [ ]:
if 'df' in locals():
    df['date'] = pd.to_datetime(df['date'])
    daily_sales = df.groupby('date')['units_sold'].sum()
    
    plt.figure(figsize=(14, 5))
    daily_sales.plot(title='Total Daily Units Sold Across All Sample Series')
    plt.ylabel('Units Sold')
    plt.xlabel('Date')
    plt.show()

## 3. Price Elasticity & Promotional Sensitivity

In [ ]:
if 'df' in locals() and 'sell_price' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df.sample(min(1000, len(df))), x='sell_price', y='units_sold', alpha=0.5)
    plt.title('Demand vs Price Scatter')
    plt.xlabel('Selling Price ($)')
    plt.ylabel('Daily Units Sold')
    plt.show()